### Rechenrätsel – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021/2022), 2. Runde, Aufgabe 2*

Gwendoline stellt ihren Mitschülern gerne Rätsel der Form `4 ∘ 4 ∘ 3 = 13`. Für „∘“ sind die Operatoren `+`, `−`, `∗`, `:`
(Addition, Subtraktion, Multiplikation, Division) einzusetzen, so dass die Rechnung stimmt. Im Beispiel wäre `4 ∗ 4 − 3 = 13` die
richtige Lösung.

Du möchtest dich revanchieren und Gwendoline richtig schwere Rätsel dieser Art stellen.

#### Aufgabe

Schreibe ein Programm, das Rätsel nach Gwendolines Schema erstellt. Die Rätsel sollten interessant und unterschiedlich sein (also
z. B. nicht `2 ∘ 2 ∘ 2 ∘ 2 = 16`). Das Programm sollte für eine gegebene Anzahl von Operatoren ein Rätsel so erzeugen, dass

a) das Rätsel eindeutig lösbar ist (also nicht `3 ∘ 4 ∘ 3 = 15`, welches zwei Lösungen hat),
b) die Operanden einzelne Ziffern sind,
c) eine positive ganze Zahl als Ergebnis herauskommt,
d) Punkt- vor Strich-Rechnung angewandt wird,
e) gleichrangige Operatoren linksassoziativ angewandt werden (also z. B. `6 : 3 ∗ 2 = 4` oder `5 − 2 + 1 = 4`) und
f) alle Zwischenergebnisse ganze Zahlen sind (also nicht `3 : 2 ∗ 4 = 6`).

Lasse dein Programm für verschiedene Operatorenanzahlen laufen und zeige uns mehrere deiner Ergebnisse.

Schaffst du es, Rätsel mit 10, 15 oder sogar noch mehr Operatoren zu generieren? Hier ist so ein Rätsel:

`4 ∘ 3 ∘ 2 ∘ 6 ∘ 3 ∘ 9 ∘ 7 ∘ 8 ∘ 2 ∘ 9 ∘ 4 ∘ 4 ∘ 6 ∘ 4 ∘ 4 ∘ 5 = 4792`

#### Beispieldaten

Es gibt keine Eingabedateien: Das Programm bekommt nur die Zahl der Operatoren. Wir prüfen es an dem Rätsel aus der Aufgabe
und an den Rätseln, die in den offiziellen Lösungshinweisen abgedruckt sind (die Ordner `beispieldaten` hat dafür die Datei
`raetsel_hinweise.txt`, von uns aus den Lösungshinweisen abgeschrieben).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Erzeugen und Prüfen:** Man würfelt die Ziffern und prüft dann, ob es ein Ergebnis gibt, das nur auf *eine* Art zu erreichen ist.
  Die Ergebniszahl `b` ist also nicht frei gewählt, sondern ergibt sich aus den Ziffern.
- **Zählen mit einem Wörterbuch (`dict`):** Für jedes erreichbare Ergebnis merken wir, auf wie viele Arten es erreichbar ist. Mehr als
  „einmal“ oder „mehr als einmal“ brauchen wir nicht zu wissen, deshalb deckeln wir die Zahl bei 2.
- **Dynamische Programmierung:** Es gibt `4^n` Belegungen der Operatoren, aber nur wenige verschiedene Zwischenergebnisse. Wir rechnen
  Stück für Stück mit der *Menge* der möglichen Zwischenergebnisse statt mit jeder Belegung einzeln.
- **Punkt vor Strich:** Ein Term wie `a ∗ b : c ∗ d` ist ein *Produktblock*; die Blöcke werden addiert oder subtrahiert.
- **Zufall:** Ziffern mit `random.choice`; mit `random.seed(...)` bleibt das Ergebnis reproduzierbar.

**Nützliche Funktionen und Bibliotheken**
- `dict` mit `.get(schluessel, 0)`, `.items()` und `min(2, …)` zum Deckeln.
- `itertools.product('+-*:', repeat=n)` für alle Belegungen (nur zum Prüfen, bei kleinem `n`).
- `random` (`seed`, `choice`) und `min(...)` über eine Liste von Ergebnissen.

## Die Idee

**Rechnen.** Zuerst brauchen wir eine Funktion `rechne(ziffern, ops)`, die ein Rätsel mit gegebenen Operatoren auswertet: Punkt vor
Strich, gleichrangige Operatoren von links, und `None`, sobald eine Division nicht aufgeht (oder durch 0 geht). Das ist leicht, wenn man
die Rechnung als Summe von **Termen** liest: Ein `+` oder `−` beginnt einen neuen Term, `∗` und `:` verändern den aktuellen Term.

**Alle Ergebnisse zählen (Brute Force).** Für kleine `n` probieren wir alle `4^n` Belegungen und zählen, wie oft jedes Ergebnis
vorkommt. Ein Rätsel mit Ergebnis `b` ist eindeutig, wenn `b` genau einmal vorkommt. So prüfen wir später alle schnelleren Verfahren.

**Schneller mit dynamischer Programmierung.** Bei 16 Operatoren sind es `4^16 ≈ 4,3 Milliarden` Belegungen; das ist zu viel. Viele davon
liefern aber dasselbe Zwischenergebnis. Wir rechnen deshalb mit zwei Tabellen von „Ergebnis → Anzahl der Arten“:

- `P[(i, j)]`: Was kann der Produktblock `a[i] ∘ … ∘ a[j]` ergeben, wenn alle `∘` dazwischen `∗` oder `:` sind?
- `S[j]`: Was kann `a[0] ∘ … ∘ a[j]` mit beliebigen Operatoren ergeben? Der letzte Produktblock reicht von `a[i]` bis `a[j]`; davor steht
  ein `+` oder `−` (oder nichts, wenn `i = 0`), und davor ein Ergebnis aus `S[i-1]`.

**Rätsel erzeugen.** Wir würfeln die Ziffern aus `2` bis `9` und nehmen als Ergebnis das **kleinste positive `b`, das genau einmal**
vorkommt. Kleine Ergebnisse sind interessanter, denn sie schließen aus, dass man einfach „alles multiplizieren“ kann, um die Größe zu
treffen. Ziffern `0` und `1` lassen wir weg, denn dort ist es oft egal, was man tut (`+ 0` oder `− 0`, `∗ 1` oder `: 1`).

## Schritt 1: Ein Rätsel auswerten und alle Belegungen zählen

`ops` ist ein String aus `+`, `-`, `*`, `:` mit einem Zeichen weniger als Ziffern. Zum Auswerten merken wir uns die Summe der
abgeschlossenen Terme `summe`, das Vorzeichen `vz` und den aktuellen Term `term`.

In [ ]:
import itertools
import random

OPERATOREN = '+-*:'


def rechne(ziffern, ops):
    """Wert von ziffern[0] ops[0] ziffern[1] ... (Punkt vor Strich, von links), None bei nicht ganzzahliger Division."""
    summe, vz, term = 0, 1, ziffern[0]
    for op, x in zip(ops, ziffern[1:]):
        if op == '*':
            term = term * x
        elif op == ':':
            if x == 0 or term % x != 0:
                return None
            term = term // x
        else:                       # + oder -: der bisherige Term ist abgeschlossen
            summe = summe + vz * term
            vz = 1 if op == '+' else -1
            term = x
    return summe + vz * term


def zaehle_brute(ziffern):
    """Dictionary Ergebnis -> Anzahl der Operatorbelegungen, alle 4^n Belegungen ausprobiert."""
    anzahl = {}
    for ops in itertools.product(OPERATOREN, repeat=len(ziffern) - 1):
        wert = rechne(ziffern, ops)
        if wert is not None:
            anzahl[wert] = anzahl.get(wert, 0) + 1
    return anzahl


print(rechne([4, 4, 3], '*-'))          # 4 * 4 - 3 = 13
print(rechne([6, 3, 2], ':*'))          # 6 : 3 * 2 = 4
print(rechne([5, 2, 1], '-+'))          # 5 - 2 + 1 = 4
print(rechne([3, 2, 4], ':*'))          # 3 : 2 ist nicht ganzzahlig
z = zaehle_brute([4, 4, 3])
print(z[13], 'Art(en) für 13;', 'alle Ergebnisse:', sorted(z.items()))
print(zaehle_brute([3, 4, 3])[15], 'Arten für 3 ∘ 4 ∘ 3 = 15')

13
4
4
None
1 Art(en) für 13; alle Ergebnisse: [(-8, 1), (-3, 1), (-2, 1), (3, 2), (4, 1), (5, 1), (11, 1), (13, 1), (16, 1), (19, 1), (48, 1)]
2 Arten für 3 ∘ 4 ∘ 3 = 15


## Schritt 2: Dynamische Programmierung

`erhoehe(tabelle, wert, anzahl)` zählt eine weitere Art, `wert` zu erreichen, hoch, aber höchstens bis 2.
Daraus bauen wir erst die Produktblöcke `P[(i, j)]` (aus `P[(i, j-1)]` durch `∗ a[j]` oder `: a[j]`), dann die Summen `S[j]`:

`S[j]` enthält zum einen die reinen Produkte `P[(0, j)]`, zum anderen für jedes `i` von 1 bis `j` alle `s + p` und `s − p` mit
`s` aus `S[i-1]` und `p` aus `P[(i, j)]`. Die Anzahlen der Arten multiplizieren sich (und werden wieder auf 2 gedeckelt).

In [ ]:
def erhoehe(tabelle, wert, anzahl):
    tabelle[wert] = min(2, tabelle.get(wert, 0) + anzahl)


def produkte(a):
    """P[(i, j)]: Dictionary Wert -> Anzahl (höchstens 2) für den Produktblock a[i] ... a[j] mit * und :."""
    P = {}
    for i in range(len(a)):
        P[(i, i)] = {a[i]: 1}
        for j in range(i + 1, len(a)):
            neu = {}
            for x, anz in P[(i, j - 1)].items():
                erhoehe(neu, x * a[j], anz)
                if a[j] != 0 and x % a[j] == 0:
                    erhoehe(neu, x // a[j], anz)
            P[(i, j)] = neu
    return P


def summen(a):
    """Gibt (P, S) zurück; S[j]: Dictionary Wert -> Anzahl (höchstens 2) für a[0] ∘ ... ∘ a[j]."""
    P = produkte(a)
    S = [{a[0]: 1}]
    for j in range(1, len(a)):
        neu = dict(P[(0, j)])
        for i in range(1, j + 1):
            for s, anz_s in S[i - 1].items():
                for p, anz_p in P[(i, j)].items():
                    erhoehe(neu, s + p, anz_s * anz_p)
                    erhoehe(neu, s - p, anz_s * anz_p)
        S.append(neu)
    return P, S


def zaehle(a):
    return summen(a)[1][-1]


print(zaehle([4, 4, 3]))                 # Tabelle der Lösungshinweise: {-8, -3, -2, 3, 3, 4, 5, 11, 13, 16, 19, 48}
P, S = summen([4, 4, 3])
print(P[(0, 2)], P[(1, 2)], S[1])

{48: 1, 3: 2, 16: 1, -8: 1, 19: 1, 13: 1, 4: 1, -2: 1, 11: 1, 5: 1, -3: 1}
{48: 1, 3: 1} {12: 1} {16: 1, 1: 1, 8: 1, 0: 1}


Die Ergebnisse stimmen mit der kleinen Tabelle der Lösungshinweise für `4 ∘ 4 ∘ 3` überein: `P(0,1) = {1, 16}`, `P(0,2) = {3, 48}`,
`P(1,2) = {12}`, `S1 = {0, 1, 8, 16}`. (Ein Wert, der zweimal vorkommt, wie die `3` in `S2`, steht bei uns einmal mit Anzahl 2; der Test, den wir brauchen,
ist „Anzahl genau 1“.)

Jetzt prüfen wir das schnelle Verfahren gegen das Durchprobieren, an vielen zufälligen kurzen Ziffernfolgen, **auch mit `0` und `1`**:

In [ ]:
random.seed(2022)
geprueft = 0
for _ in range(300):
    n = random.randint(1, 7)
    a = [random.randint(0, 9) for _ in range(n + 1)]
    brute = {wert: min(2, anz) for wert, anz in zaehle_brute(a).items()}
    assert brute == zaehle(a), a
    geprueft += 1
print(geprueft, 'zufällige Ziffernfolgen: Tabellenverfahren und Durchprobieren stimmen überein')

300 zufällige Ziffernfolgen: Tabellenverfahren und Durchprobieren stimmen überein


## Schritt 3: Das kleinste eindeutige Ergebnis

Ein Rätsel ist eindeutig, wenn das Ergebnis `b` in der Tabelle `S[n]` mit Anzahl **1** steht. Wir nehmen das kleinste positive solche `b`.
Als Probe das Rätsel aus der Aufgabe: Die Lösungshinweise nennen als die 14 kleinsten eindeutigen Ergebnisse
`4792, 5118, 5136, 5138, 5142, 5152, 5156, 5163, 5167, 5168, 5194, 5200, 5213, 5215`.

In [ ]:
def eindeutige_ergebnisse(a):
    """Alle positiven Ergebnisse, die auf genau eine Art erreichbar sind, aufsteigend."""
    return sorted(b for b, anz in zaehle(a).items() if anz == 1 and b > 0)


def kleinstes_eindeutiges(a):
    liste = eindeutige_ergebnisse(a)
    return liste[0] if liste else None


AUFGABE = [4, 3, 2, 6, 3, 9, 7, 8, 2, 9, 4, 4, 6, 4, 4, 5]
print(eindeutige_ergebnisse(AUFGABE)[:14])
print(kleinstes_eindeutiges([4, 4, 3]), 'ist das kleinste eindeutige Ergebnis für 4 ∘ 4 ∘ 3')

[4792, 5118, 5136, 5138, 5142, 5152, 5156, 5163, 5167, 5168, 5194, 5200, 5213, 5215]
4 ist das kleinste eindeutige Ergebnis für 4 ∘ 4 ∘ 3


## Schritt 4: Die Lösung aus den Tabellen lesen

Zum Rätsel gehört (für die Lösung zum Nachprüfen) auch die Belegung der Operatoren. Wir lesen sie **rückwärts** aus den Tabellen:
`S[j]` enthält `b`, entweder als reines Produkt (dann `produkt_ops`) oder als `s ± p` mit `s` in `S[i-1]` und `p` in `P[(i, j)]`.
Die Tabellen enthalten nur *erreichbare* Werte, also führt jeder gefundene Teilschritt zum Ziel.

In [ ]:
def produkt_ops(a, P, i, j, x):
    """Operatoren (String aus * und :) für den Block a[i..j] mit Wert x."""
    if i == j:
        return ''
    if x % a[j] == 0 and x // a[j] in P[(i, j - 1)]:
        return produkt_ops(a, P, i, j - 1, x // a[j]) + '*'
    return produkt_ops(a, P, i, j - 1, x * a[j]) + ':'


def finde_ops(a, b):
    """Eine Operatorbelegung (String) mit Ergebnis b, oder None."""
    P, S = summen(a)

    def suche(j, wert):             # Operatoren für a[0..j] mit Ergebnis wert
        if wert in P[(0, j)]:
            return produkt_ops(a, P, 0, j, wert)
        for i in range(1, j + 1):
            for p in P[(i, j)]:
                for op, s in (('+', wert - p), ('-', wert + p)):
                    if s in S[i - 1]:
                        return suche(i - 1, s) + op + produkt_ops(a, P, i, j, p)
        return None

    return suche(len(a) - 1, b) if b in S[-1] else None


def als_text(a, ops=None, b=None):
    """Rätsel (mit ∘ oder mit eingesetzten Operatoren) als Text."""
    zeichen = {'*': '∗', '-': '−', '+': '+', ':': ':'}
    teile = [str(a[0])]
    for k, x in enumerate(a[1:]):
        teile += [zeichen[ops[k]] if ops else '∘', str(x)]
    return ' '.join(teile) + (' = %d' % b if b is not None else '')


ops = finde_ops(AUFGABE, 4792)
print(als_text(AUFGABE, None, 4792))
print(als_text(AUFGABE, ops, 4792))
assert rechne(AUFGABE, ops) == 4792

4 ∘ 3 ∘ 2 ∘ 6 ∘ 3 ∘ 9 ∘ 7 ∘ 8 ∘ 2 ∘ 9 ∘ 4 ∘ 4 ∘ 6 ∘ 4 ∘ 4 ∘ 5 = 4792
4 ∗ 3 ∗ 2 ∗ 6 ∗ 3 ∗ 9 + 7 ∗ 8 : 2 ∗ 9 ∗ 4 − 4 ∗ 6 − 4 ∗ 4 ∗ 5 = 4792


## Schritt 5: Rätsel erzeugen

Die Ziffern würfeln wir aus `2` bis `9`. Die Lösungshinweise wählen kleine Ziffern häufiger (die Rechnung bleibt dann kleiner und das Ergebnis
auch) und geben dafür empirisch bestimmte Wahrscheinlichkeiten an: `2`: 27,0 %, `3`: 21,6 %, `4`: 16,2 %, `5`: 10,8 %, `6`: 8,1 %, `7`: 6,5 %,
`8`: 5,4 %, `9`: 4,3 %. Wir bilden das mit einer Liste nach, in der jede Ziffer so oft steht, wie ihre Wahrscheinlichkeit in Prozent
angibt (gerundet). Danach prüfen wir die Rätsel: Die Lösung stimmt, und bei kleinem `n` ist das Rätsel wirklich eindeutig (durch Durchprobieren).

In [ ]:
GEWICHT = {2: 27, 3: 22, 4: 16, 5: 11, 6: 8, 7: 7, 8: 5, 9: 4}
LOS = [z for z, g in GEWICHT.items() for _ in range(g)]


def neues_raetsel(n):
    """Zufällige Ziffern (n + 1 Stück) und das kleinste eindeutige Ergebnis. Bei keinem eindeutigen Ergebnis: neu würfeln."""
    while True:
        a = [random.choice(LOS) for _ in range(n + 1)]
        b = kleinstes_eindeutiges(a)
        if b is not None:
            return a, b


random.seed(40)
raetsel = {}
for n in (3, 5, 8, 10, 12, 15):
    a, b = neues_raetsel(n)
    ops = finde_ops(a, b)
    assert rechne(a, ops) == b
    if n <= 8:
        assert zaehle_brute(a)[b] == 1            # wirklich eindeutig
    raetsel[n] = (a, b, ops)
    print('%2d Operatoren: %s' % (n, als_text(a, None, b)))
    print('    Lösung:     %s' % als_text(a, ops, b))

 3 Operatoren: 4 ∘ 5 ∘ 5 ∘ 2 = 1
    Lösung:     4 − 5 : 5 − 2 = 1
 5 Operatoren: 3 ∘ 3 ∘ 7 ∘ 6 ∘ 7 ∘ 2 = 11
    Lösung:     3 : 3 + 7 − 6 + 7 + 2 = 11
 8 Operatoren: 2 ∘ 9 ∘ 3 ∘ 3 ∘ 8 ∘ 4 ∘ 2 ∘ 5 ∘ 8 = 217
    Lösung:     2 ∗ 9 ∗ 3 + 3 + 8 : 4 ∗ 2 ∗ 5 ∗ 8 = 217
10 Operatoren: 2 ∘ 2 ∘ 6 ∘ 2 ∘ 6 ∘ 7 ∘ 4 ∘ 2 ∘ 2 ∘ 6 ∘ 3 = 386
    Lösung:     2 + 2 ∗ 6 + 2 ∗ 6 ∗ 7 + 4 ∗ 2 ∗ 2 ∗ 6 ∗ 3 = 386
12 Operatoren: 2 ∘ 3 ∘ 3 ∘ 2 ∘ 5 ∘ 7 ∘ 5 ∘ 2 ∘ 4 ∘ 2 ∘ 3 ∘ 8 ∘ 9 = 809
    Lösung:     2 ∗ 3 ∗ 3 ∗ 2 ∗ 5 + 7 ∗ 5 ∗ 2 ∗ 4 ∗ 2 − 3 + 8 ∗ 9 = 809
15 Operatoren: 3 ∘ 3 ∘ 8 ∘ 2 ∘ 3 ∘ 2 ∘ 5 ∘ 3 ∘ 7 ∘ 3 ∘ 4 ∘ 2 ∘ 5 ∘ 2 ∘ 4 ∘ 4 = 2322
    Lösung:     3 ∗ 3 ∗ 8 + 2 − 3 + 2 + 5 + 3 ∗ 7 : 3 ∗ 4 ∗ 2 ∗ 5 ∗ 2 ∗ 4 + 4 = 2322


## Schritt 6: Welche Rätsel sind „interessant“?

Die Aufgabe verlangt nicht nur Eindeutigkeit. Zwei Beispiele für eindeutige, aber langweilige Rätsel:

- **Das Produkt aller Ziffern als Ergebnis** ist ab zwei Operatoren immer eindeutig (nichts anderes kommt so hoch), und die Lösung ist
  „überall `∗`“. Deshalb nehmen wir nicht ein beliebiges eindeutiges `b`, sondern das **kleinste**.
- **Zahlentheorie:** `9 ∘ 9 ∘ 3 ∘ 1 = 83` hat nur die Lösung `9 ∗ 9 + 3 − 1`; man sieht es mit einer Größenbetrachtung (`: 9` wäre viel zu klein)
  und Restbetrachtungen (mod 3, mod 9), ohne zu probieren. So ein Rätsel lässt sich schnell von Hand lösen. Die Lösungshinweise wollen solche Rätsel
  vermeiden und erzeugen Rätsel, bei denen man eher Brute Force braucht. Bei uns ist das Ergebnis `b` das kleinste eindeutige; für die
  gleichen Ziffern `9 ∘ 9 ∘ 3 ∘ 1` ist das `7`, nicht `83`. Ob ein erzeugtes Rätsel dennoch mit Größen- oder Restbetrachtungen lösbar ist, prüfen wir nicht.

In [ ]:
produkt = 1
for x in [2, 3, 4, 5]:
    produkt *= x
print('Produkt', produkt, ': Anzahl der Arten', zaehle([2, 3, 4, 5]).get(produkt), '(nur Mal-Mal-Mal)')
print('2 ∘ 2 ∘ 2 ∘ 2 = 16:', zaehle_brute([2, 2, 2, 2])[16], 'Art(en)')
print('9 ∘ 9 ∘ 3 ∘ 1 = 83:', zaehle([9, 9, 3, 1])[83], '(gedeckelt) ->', finde_ops([9, 9, 3, 1], 83))
print('kleinstes eindeutiges Ergebnis für 9 ∘ 9 ∘ 3 ∘ 1:', kleinstes_eindeutiges([9, 9, 3, 1]),
      finde_ops([9, 9, 3, 1], kleinstes_eindeutiges([9, 9, 3, 1])))

Produkt 120 : Anzahl der Arten 1 (nur Mal-Mal-Mal)
2 ∘ 2 ∘ 2 ∘ 2 = 16: 1 Art(en)
9 ∘ 9 ∘ 3 ∘ 1 = 83: 1 (gedeckelt) -> *+-
kleinstes eindeutiges Ergebnis für 9 ∘ 9 ∘ 3 ∘ 1: 7 -:+


Auch der Verzicht auf die Ziffern `0` und `1` lässt sich messen. Wir würfeln jeweils 200 Ziffernfolgen mit 8 Operatoren und zählen:
wie oft es überhaupt ein eindeutiges Ergebnis gibt, und wie groß das kleinste im Mittel ist. Die drei Ziffernmengen: `0`–`9` gleichverteilt,
`2`–`9` gleichverteilt, `2`–`9` gewichtet wie in den Hinweisen.

In [ ]:
def statistik(erzeuge_ziffer, n=8, versuche=200):
    mit, summe_b = 0, 0
    for _ in range(versuche):
        a = [erzeuge_ziffer() for _ in range(n + 1)]
        b = kleinstes_eindeutiges(a)
        if b is not None:
            mit += 1
            summe_b += b
    return mit, round(summe_b / max(mit, 1))


random.seed(8)
for name, erzeuge in [('0 bis 9 gleichverteilt', lambda: random.randint(0, 9)),
                      ('2 bis 9 gleichverteilt', lambda: random.randint(2, 9)),
                      ('2 bis 9 gewichtet', lambda: random.choice(LOS))]:
    mit, mittel = statistik(erzeuge)
    print('%-24s Folgen mit eindeutigem Ergebnis: %3d von 200, kleinstes b im Mittel: %d' % (name, mit, mittel))

0 bis 9 gleichverteilt   Folgen mit eindeutigem Ergebnis: 101 von 200, kleinstes b im Mittel: 162
2 bis 9 gleichverteilt   Folgen mit eindeutigem Ergebnis: 200 von 200, kleinstes b im Mittel: 187
2 bis 9 gewichtet        Folgen mit eindeutigem Ergebnis: 200 von 200, kleinstes b im Mittel: 149


Mit den Ziffern `0` und `1` hat nur etwa jede zweite Folge überhaupt ein eindeutiges positives Ergebnis, mit `2` bis `9` jede (in diesen Proben).
Bei der gewichteten Wahl ist das kleinste `b` im Mittel kleiner als bei gleichverteilten Ziffern (so sagen es auch die Hinweise); dafür kommen
`2` und `3` häufig vor, die Rätsel sind also weniger abwechslungsreich.

## Alle Beispiele: Rätsel aus den Lösungshinweisen und das Rätsel der Aufgabe

Die Lösungshinweise geben Rätsel mit 3 bis 20 Operatoren an. Wir schreiben ihre Ziffern in eine Datei `beispieldaten/raetsel_hinweise.txt`
(Zeile: Ziffern, dahinter das Ergebnis; **von uns aus den Hinweisen abgeschrieben**, die Aufgabe selbst hat keine Beispieldateien) und
fragen, ob unser Verfahren zu denselben Ziffern dasselbe kleinste eindeutige Ergebnis findet.

In [ ]:
def lies_raetsel(dateiname):
    liste = []
    with open(dateiname, encoding='utf-8') as datei:
        for zeile in datei:
            zeile = zeile.strip()
            if zeile and not zeile.startswith('#'):
                teile = zeile.split('=')
                liste.append(([int(x) for x in teile[0].split()], int(teile[1])))
    return liste


HINWEISE = lies_raetsel('beispieldaten/raetsel_hinweise.txt')
ERGEBNIS = []
for a, b_hinweis in HINWEISE:
    P, S = summen(a)
    b = kleinstes_eindeutiges(a)
    eintraege = sum(len(tabelle) for tabelle in S) + sum(len(tabelle) for tabelle in P.values())
    ERGEBNIS.append((len(a) - 1, b_hinweis, b, S[-1].get(b_hinweis), eintraege))
    werte = (len(a) - 1, b_hinweis, b, S[-1].get(b_hinweis), eintraege)
    print('%2d Operatoren: Hinweise %6d, wir %6d, Anzahl der Arten für das Ergebnis der Hinweise: %s, Tabelleneinträge: %d' % werte)

 3 Operatoren: Hinweise      6, wir      6, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 49
 5 Operatoren: Hinweise     11, wir     11, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 304
 8 Operatoren: Hinweise    110, wir    110, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 1185
10 Operatoren: Hinweise    353, wir    353, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 7844
12 Operatoren: Hinweise   1521, wir   1521, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 45255
15 Operatoren: Hinweise   6295, wir   6295, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 165760
18 Operatoren: Hinweise  17464, wir  15085, Anzahl der Arten für das Ergebnis der Hinweise: None, Tabelleneinträge: 536128
20 Operatoren: Hinweise  19330, wir  19330, Anzahl der Arten für das Ergebnis der Hinweise: 1, Tabelleneinträge: 651735


Bei 3, 5, 8, 10, 12, 15 und 20 Operatoren finden wir dasselbe Ergebnis wie die Lösungshinweise. Bei 18 Operatoren steht dort `17464`. Für
genau diese Ziffern ist `17464` in unserer Tabelle **gar nicht erreichbar** (Anzahl `None`); unser kleinstes eindeutiges Ergebnis ist
`15085`. Wir haben die Ziffern genau so aus den Hinweisen abgeschrieben, wie sie dort stehen, und ein Test mit dem Durchprobieren
bestätigt unser Verfahren an vielen anderen Folgen (siehe oben). Vermutlich ist in den Hinweisen eine Ziffer falsch abgedruckt; wir
können es aber nicht nachprüfen. Auch das Rätsel der Aufgabe stimmt: Seine 14 kleinsten eindeutigen Ergebnisse (Schritt 3) sind
genau die Liste der Hinweise.

Wie groß kann `n` werden? Die Zahl der Tabelleneinträge (letzte Spalte) wächst schneller als `n`, aber lange nicht wie `4^n`:

In [ ]:
for n_op, _, _, _, eintraege in ERGEBNIS:
    print('%2d Operatoren: %7d Tabelleneinträge (Belegungen: 4^%d = %d)' % (n_op, eintraege, n_op, 4 ** n_op))
a15 = [a for a, _ in HINWEISE if len(a) == 16][0]
eind15 = eindeutige_ergebnisse(a15)
print('15 Operatoren: %d verschiedene eindeutige Ergebnisse, die kleinsten: %s' % (len(eind15), eind15[:6]))

 3 Operatoren:      49 Tabelleneinträge (Belegungen: 4^3 = 64)
 5 Operatoren:     304 Tabelleneinträge (Belegungen: 4^5 = 1024)
 8 Operatoren:    1185 Tabelleneinträge (Belegungen: 4^8 = 65536)
10 Operatoren:    7844 Tabelleneinträge (Belegungen: 4^10 = 1048576)
12 Operatoren:   45255 Tabelleneinträge (Belegungen: 4^12 = 16777216)
15 Operatoren:  165760 Tabelleneinträge (Belegungen: 4^15 = 1073741824)
18 Operatoren:  536128 Tabelleneinträge (Belegungen: 4^18 = 68719476736)
20 Operatoren:  651735 Tabelleneinträge (Belegungen: 4^20 = 1099511627776)
15 Operatoren: 6867 verschiedene eindeutige Ergebnisse, die kleinsten: [6295, 6324, 6330, 6342, 6376, 6377]


So kommen wir auch mit 20 Operatoren noch ans Ziel. Die Hinweise nennen ca. 4000 bis 8000 verschiedene eindeutige Ergebnisse `b` ab
15 Operatoren; bei der Ziffernfolge mit 15 Operatoren von oben zählen wir 6867 (letzte Ausgabezeile), das passt dazu.

## Zum Schluss: Mit MiniZinc

Mit MiniZinc kann man die **Prüfung** sehr kurz formulieren: Gegeben sind die Ziffern und `b`; gefragt ist, mit welchen Operatoren es
stimmt. Lässt man **alle** Lösungen ausgeben (Option „Alle Lösungen ausgeben“ in der IDE, auf der Kommandozeile `-a`) und zählt sie,
weiß man, ob ein Rätsel eindeutig ist. Du kennst von den Arbeitsblättern Arrays, `forall`, Variablen als Index und `.dzn`-Dateien. Neu ist
hier, dass wir den Rechenweg als **Zustand** von Schritt zu Schritt aufschreiben, genau wie `rechne`:

- `o[k]` ist der Operator vor Ziffer `k` (1: `+`, 2: `−`, 3: `∗`, 4: `:`).
- `summe[k]`, `vz[k]` und `term[k]` sind die Werte nach `k` Schritten (wie in `rechne`). Bei `+` oder `−` wird der alte Term zur Summe gezählt und ein
  neuer Term beginnt. Bei `∗` wächst der Term, bei `:` verlangt `term mod a = 0`, dass die Division aufgeht.
- Die Solver rechnen mit 32-Bit-Zahlen. Damit nichts überläuft, geben wir eine Obergrenze `G` mit an (das Produkt aller Ziffern; kein Term
  kann größer sein). Das reicht daher nur für Rätsel mit kleinem Ziffernprodukt.

```minizinc
int: n;                                   % Anzahl der Operatoren
int: b;                                   % das Ergebnis des Rätsels
int: G;                                   % Obergrenze für jeden Term (Produkt aller Ziffern)
array[0..n] of int: a;                    % die Ziffern
array[1..n] of var 1..4: o;               % 1: +   2: -   3: *   4: :
array[0..n] of var 0..G: term;            % aktueller Term (Produktblock)
array[0..n] of var {-1, 1}: vz;           % Vorzeichen des aktuellen Terms
array[0..n] of var -(n + 1) * G..(n + 1) * G: summe;   % Summe der abgeschlossenen Terme

constraint term[0] = a[0] /\ vz[0] = 1 /\ summe[0] = 0;
constraint forall(k in 1..n)(
  if o[k] <= 2 then                       % + oder -: neuer Term beginnt
    summe[k] = summe[k-1] + vz[k-1] * term[k-1] /\ term[k] = a[k] /\ vz[k] = (if o[k] = 1 then 1 else -1 endif)
  elseif o[k] = 3 then                    % *: Term wird multipliziert
    term[k-1] <= G div a[k] /\ term[k] = term[k-1] * a[k] /\ summe[k] = summe[k-1] /\ vz[k] = vz[k-1]
  else                                    % :: die Division muss aufgehen
    term[k-1] mod a[k] = 0 /\ term[k] = term[k-1] div a[k] /\ summe[k] = summe[k-1] /\ vz[k] = vz[k-1]
  endif);
constraint summe[n] + vz[n] * term[n] = b;

solve satisfy;
output ["\(o)\n"];
```

Die Datei mit den Daten erzeugt die Python-Zelle `als_dzn`. Das Array `a` beginnt bei Index 0, deshalb `array1d(0..n, …)`.

In [ ]:
def als_dzn(a, b):
    produkt = 1
    for x in a:
        produkt *= x
    return 'n = %d;\nb = %d;\nG = %d;\na = array1d(0..%d, %s);' % (len(a) - 1, b, produkt, len(a) - 1, a)


print(als_dzn([4, 4, 3], 13))

n = 2;
b = 13;
G = 48;
a = array1d(0..2, [4, 4, 3]);


Ergebnis beim Erzeugen dieses Notebooks (Zeitlimit 40 Sekunden pro Aufruf, Gecode und Chuffed; in MiniZinc über *Solver configuration*
gewählt). Die Spalten zeigen die Zahl der Lösungen, die der Solver findet (Python deckelt bei 2, „2“ heißt dort „mindestens zwei“), „?“ heißt:
Zeitlimit, „–“: die Obergrenze `G` passt nicht in 32 Bit. Die Zahl stimmt jeweils mit dem Durchprobieren in Python überein
(`zaehle_brute`; dort, wo `n ≤ 10` ist).

| Rätsel | Operatoren | Python (gedeckelt) | Gecode | Chuffed |
|--------|------------|--------------------|--------|---------|
| 4 ∘ 4 ∘ 3 = 13 | 2 | 1 | 1 | 1 |
| 3 ∘ 4 ∘ 3 = 15 | 2 | 2 | 2 | 2 |
| eigenes Rätsel, 3 Operatoren | 3 | 1 | 1 | 1 |
| eigenes Rätsel, 5 Operatoren | 5 | 1 | 1 | 1 |
| eigenes Rätsel, 8 Operatoren | 8 | 1 | 1 | 1 |
| eigenes Rätsel, 10 Operatoren | 10 | 1 | 1 | 1 |
| eigenes Rätsel, 12 Operatoren | 12 | 1 | 1 | 1 |
| eigenes Rätsel, 15 Operatoren | 15 | – | – | – |
| Rätsel der Hinweise, 3 Operatoren | 3 | 1 | 1 | 1 |
| Rätsel der Hinweise, 5 Operatoren | 5 | 1 | 1 | 1 |
| Rätsel der Hinweise, 8 Operatoren | 8 | 1 | 1 | 1 |
| Rätsel der Hinweise, 10 Operatoren | 10 | 1 | 1 | 1 |
| Rätsel der Hinweise, 12 Operatoren | 12 | – | – | – |

Das Modell prüft Eindeutigkeit richtig, und Chuffed schafft auch die Rätsel mit 12 Operatoren schnell (Gecode braucht merklich länger). Die Grenze setzt
die 32-Bit-Arithmetik: Bei größeren Ziffernprodukten (ab etwa 15 Operatoren oder bei größeren Ziffern, wie bei den Rätseln der Hinweise)
läuft das Modell so nicht. Außerdem fasst der Solver gleiche Zwischenergebnisse nicht zusammen wie unsere Tabellen `S` und `P`, sondern
durchsucht im Prinzip die Belegungen, nur mit Abkürzungen. Zum **Erzeugen** der Rätsel taugt MiniZinc hier nicht: Wir müssen das
Ergebnis `b` vorher wissen, und die Suche nach dem kleinsten eindeutigen `b` wäre ein Aufruf pro Kandidat.

## Für Neugierige

- **Warum klappt die Tabelle so gut?** Es gibt nur wenige verschiedene Zwischenergebnisse (Tabelleneinträge oben). Im schlimmsten Fall
  bleibt es bei `O(4^n · n)` wie beim Durchprobieren; es werden aber nie mehr Rechnungen gemacht, meist viel weniger.
- **Kein eindeutiges Ergebnis:** In unseren Proben mit Ziffern `2` bis `9` gab es immer eines. Sollte eine Folge keines haben,
  würfelt `neues_raetsel` einfach neu.
- **Ziffern 0 und 1 zulassen:** Dann wird es schwerer, ein *interessantes* eindeutiges Ergebnis zu finden (siehe Statistik oben); `0` und `1` machen
  viele Belegungen gleichwertig.
- **Größere `n`:** Ab 20 Operatoren wird die Rechnung in Python merklich langsamer, und die Tabellen wachsen weiter (siehe oben). Die Hinweise
  sparen Rechenzeit durch kleine Ziffern, aufeinanderfolgende möglichst teilerfremd; bei etwa 20 Operatoren läuft ihr Programm so ungefähr doppelt
  so schnell wie bei gleichverteilten Ziffern.
- **Ein Rätsel lösen:** `finde_ops(a, b)` löst ein gegebenes Rätsel mit denselben Tabellen, ohne alle `4^n` Belegungen zu probieren.